In [4]:
from scipy.optimize import minimize
import numpy as np

# 1. YOUR CLASSICAL SOLUTION
# (Replace this random array with the actual 180-bit list from your classical algorithm)
classical_bitstring = np.random.randint(0, 2, num_nodes) 

# Convert bits (0, 1) into target expectation values (-1.0, 1.0)
target_expectations = np.array([1.0 if bit == 1 else -1.0 for bit in classical_bitstring])

print("Starting Supervised Pre-training to embed the classical cut...")

# 2. THE DUMMY MSE LOSS FUNCTION
def pretrain_mse_loss(x):
    # Run the estimator using your exact same ansatz and observables
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    # Extract the expectation values
    idx = 0
    node_exp_map = {}
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    # Calculate Mean Squared Error against the classical target
    mse_loss = 0
    for i in range(num_nodes):
        mse_loss += (node_exp_map[i] - target_expectations[i]) ** 2
        
    return mse_loss / num_nodes

# 3. PRE-TRAIN THE CIRCUIT WITH L-BFGS-B
# L-BFGS-B is perfect for this because it minimizes simple MSE functions incredibly fast
np.random.seed(42)
random_start = np.random.rand(ansatz.num_parameters)

pretrain_result = minimize(
    fun=pretrain_mse_loss, 
    x0=random_start,
    method="L-BFGS-B", 
    options={"maxiter": 150, "disp": True}
)

# 4. THE HANDOFF
classical_warm_start_params = pretrain_result.x
print(f"Pre-training complete! Final MSE: {pretrain_result.fun:.6f}")
print("The quantum circuit has successfully memorized the classical cut.")

# Save it to disk so your main optimization script can load it!
np.save("pce_optimal_theta_CLASSICAL_WARMSTART.npy", classical_warm_start_params)

Starting Supervised Pre-training to embed the classical cut...


NameError: name 'ansatz' is not defined

In [3]:
import pandas as pd
import numpy as np
import networkx as nx
from networkx.algorithms.approximation import maxcut

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())

# ==========================================
# 2. CLASSICAL LOCAL SEARCH (ONE-EXCHANGE)
# ==========================================
best_cut_value = 0
best_partition = None

print("Running classical one_exchange heuristic with 100 random restarts...")

# Run 100 times to guarantee we don't get trapped in a bad local minimum
for i in range(1):
    # Create a random 50/50 split to start
    nodes = list(graph.nodes())
    np.random.shuffle(nodes)
    half = len(nodes) // 2
    
    # NetworkX only needs to know ONE side of the initial cut
    initial_cut_side = set(nodes[:half]) 
    
    # Run the NetworkX one_exchange algorithm
    cut_value, partition = maxcut.one_exchange(
        graph, 
        initial_cut=initial_cut_side,  # <--- FIXED ARGUMENT NAME
        weight='weight'
    )
    
    # Track the absolute best run
    if cut_value > best_cut_value:
        best_cut_value = cut_value
        best_partition = partition

print(f"\n✅ Best Classical Cut Found: {best_cut_value}")

# ==========================================
# 3. EXTRACT TARGET BITSTRING
# ==========================================
# Initialize an array of zeros
classical_bitstring = np.zeros(num_nodes, dtype=int)

# NetworkX returns a tuple of two sets: (Set_A, Set_B)
set_A, set_B = best_partition

# Map Set A to '1' and Set B to '0'
for node in set_A:
    classical_bitstring[node] = 1
for node in set_B:
    classical_bitstring[node] = 0

print("\nExtracted Target Bitstring:")
print(classical_bitstring)

# Save the classical bitstring to your hard drive
np.save("classical_target_bitstring.npy", classical_bitstring)
print("\nSaved to 'classical_target_bitstring.npy'. Ready for Quantum Pre-training!")

Loading dataset...
Running classical one_exchange heuristic with 100 random restarts...

✅ Best Classical Cut Found: 6407.2908228076385

Extracted Target Bitstring:
[0 1 0 1 1 1 0 0 1 0 0 0 1 1 1 1 0 1 0 0 0 1 0 1 0 0 1 1 1 1 0 1 1 1 0 0 0
 0 1 0 1 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 1 1 1 0 1 0 1 1 0 0 0 1 0 0 1 0
 1 1 1 1 1 0 1 1 0 0 0 0 1 1 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 1 1 1 1 1 1 0 1
 1 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 1 0 1 0 1 0 1 0 1 1 0 1 1 1 0 1 0
 0 1 1 0 1 1 1 0 0 0 0 0 1 1 0 0 0 1 0 0 1 1 1 0 0 0 1 0 1 1 1 0]

Saved to 'classical_target_bitstring.npy'. Ready for Quantum Pre-training!


In [5]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import minimize

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
# Using the exact noiseless Estimator for perfect pre-training
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET (For dimensions)
# ==========================================
print("Loading dataset dimensions...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
# Must match your main VQE script perfectly!
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = Estimator()

# ==========================================
# 4. LOAD CLASSICAL TARGET & DEFINE MSE LOSS
# ==========================================
target_file = "classical_target_bitstring.npy"

if not os.path.exists(target_file):
    raise FileNotFoundError(f"Could not find {target_file}. Run the classical NetworkX script first!")

classical_bitstring = np.load(target_file)
print("Successfully loaded classical target bitstring.")

# Convert 0s and 1s into quantum expectation targets of -1.0 and 1.0
target_expectations = np.array([1.0 if bit == 1 else -1.0 for bit in classical_bitstring])

def pretrain_mse_loss(x):
    # Run the exact simulator
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    # Extract expectation values
    idx = 0
    node_exp_map = {}
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    # Calculate Mean Squared Error (MSE)
    mse_loss = 0
    for i in range(num_nodes):
        mse_loss += (node_exp_map[i] - target_expectations[i]) ** 2
        
    return mse_loss / num_nodes

# ==========================================
# 5. EXECUTE L-BFGS-B PRE-TRAINING
# ==========================================
print("\nStarting Supervised Pre-training to embed the classical cut...")
np.random.seed(42)
initial_guess = np.random.rand(ansatz.num_parameters)

# L-BFGS-B is perfectly suited for fast MSE convergence
pretrain_result = minimize(
    fun=pretrain_mse_loss, 
    x0=initial_guess,
    method="L-BFGS-B", 
    options={"maxiter": 200, "disp": True}
)

classical_warm_start_params = pretrain_result.x
print(f"\n✅ Pre-training complete! Final MSE: {pretrain_result.fun:.6f}")

if pretrain_result.fun < 0.1:
    print("The quantum circuit has successfully memorized the classical cut!")
else:
    print("Warning: The MSE is a bit high. The circuit may need more 'reps' to fully express this specific cut.")

# ==========================================
# 6. SAVE WARM-START PARAMETERS
# ==========================================
save_filename = "pce_optimal_theta_CLASSICAL_WARMSTART.npy"
np.save(save_filename, classical_warm_start_params)
print(f"Saved parameters to '{save_filename}'.")
print("You can now feed this file into your main Adam/SPSA VQE script!")

Loading dataset dimensions...
Successfully loaded classical target bitstring.

Starting Supervised Pre-training to embed the classical cut...


KeyboardInterrupt: 

In [10]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp

# Explicitly using EstimatorV2 from Qiskit Aer
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET (For dimensions)
# ==========================================
print("Loading dataset dimensions...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATORV2 SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=6)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

# Initialize Aer EstimatorV2
estimator = EstimatorV2()

# ⚠️ FOR PRE-TRAINING WITH ADAM: We leave shots as exact/None. 
# If you set default_shots = 10000 here, approx_fprime will bottleneck and crash the gradient math!
# estimator.options.default_shots = 10000 

experiment_result = []

# ==========================================
# 4. LOAD CLASSICAL TARGET & DEFINE MSE LOSS
# ==========================================
target_file = "classical_target_bitstring.npy"

if not os.path.exists(target_file):
    raise FileNotFoundError(f"Could not find {target_file}. Run the classical NetworkX script first!")

classical_bitstring = np.load(target_file)
print("Successfully loaded classical target bitstring.")

# Convert 0s and 1s into quantum expectation targets of -1.0 and 1.0
target_expectations = np.array([1.0 if bit == 1 else -1.0 for bit in classical_bitstring])

def pretrain_mse_loss(x, silent=False):
    # EstimatorV2 uses Primitive Unified Blocs (PUBs)
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    idx = 0
    node_exp_map = {}
    
    # Iterate through the V2 PubResults
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    # Calculate Mean Squared Error (MSE)
    mse_loss = 0
    for i in range(num_nodes):
        mse_loss += (node_exp_map[i] - target_expectations[i]) ** 2
        
    mse_loss = mse_loss / num_nodes
    
    if not silent:
        experiment_result.append(mse_loss)
        print(f"Iter {len(experiment_result)}: MSE Loss = {mse_loss:.6f}")
        
    return mse_loss

def silent_pretrain_mse_loss(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return pretrain_mse_loss(x, silent=True)

# ==========================================
# 5. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=200):
    print(f"\nStarting Supervised Pre-training with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    best_overall_loss = float('inf')
    best_overall_theta = np.copy(theta_i)
    
    current_loss = pretrain_mse_loss(theta_i, silent=False)
    if current_loss < best_overall_loss:
        best_overall_loss = current_loss
        best_overall_theta = np.copy(theta_i)
        
    for i in range(1, max_iter + 1):
        print(f"\n--- Adam Step {i}/{max_iter} ---")
        t += 1
        
        # Calculates the Euclidean gradient quietly
        grad = approx_fprime(theta_i, silent_pretrain_mse_loss, epsilon=1e-5)
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            current_loss = pretrain_mse_loss(theta_i, silent=False)
            if current_loss < best_overall_loss:
                best_overall_loss = current_loss
                best_overall_theta = np.copy(theta_i)
            break
            
        theta_i = theta_next
        current_loss = pretrain_mse_loss(theta_i, silent=False)
        
        if current_loss < best_overall_loss:
            best_overall_loss = current_loss
            best_overall_theta = np.copy(theta_i)
            
    print(f"\nAdam Pre-training Complete! Absolute Best MSE Found: {best_overall_loss:.6f}")
    return best_overall_theta

# ==========================================
# 6. EXECUTE & SAVE WARM-START PARAMETERS
# ==========================================
np.random.seed(42)
initial_guess = np.random.rand(ansatz.num_parameters)

# MSE landscapes are typically much smoother than MaxCut tanh landscapes,
# allowing for a higher learning rate (0.1) to find the classical cut quickly.
classical_warm_start_params = adam_optimize(
    loss_fn=silent_pretrain_mse_loss, 
    theta0=initial_guess,
    lr=0.8,        
    max_iter=50 
)

if experiment_result[-1] < 0.1:
    print("\n✅ The quantum circuit has successfully memorized the classical cut!")
else:
    print("\n⚠️ Warning: The MSE is a bit high. The circuit may need more 'reps' to fully express this specific cut.")

save_filename = "pce_optimal_theta_CLASSICAL_WARMSTART.npy"
np.save(save_filename, classical_warm_start_params)
print(f"Saved parameters to '{save_filename}'.")
print("You can now feed this file into your main VQE script!")

Loading dataset dimensions...
Successfully loaded classical target bitstring.

Starting Supervised Pre-training with Adam (lr=0.8)...
Iter 1: MSE Loss = 0.995292

--- Adam Step 1/50 ---
Iter 2: MSE Loss = 0.991407

--- Adam Step 2/50 ---
Iter 3: MSE Loss = 1.002927

--- Adam Step 3/50 ---


KeyboardInterrupt: 

In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE) SETUP
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pce = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pce.append("".join(paulis))
        
    return [SparsePauliOp.from_list([(p, 1)]) for p in pce]

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

# Use exact noiseless Estimator to evaluate the raw learned state
estimator = EstimatorV2()

# ==========================================
# 4. LOAD PRE-TRAINED PARAMETERS
# ==========================================
param_file = "pce_optimal_theta_adam_reps4qbt12Continue (2).npy"

if not os.path.exists(param_file):
    raise FileNotFoundError(f"Cannot find {param_file}. Run the pre-training script first!")

optimal_theta = np.load(param_file)
print(f"Successfully loaded {len(optimal_theta)} parameters from {param_file}.")

# ==========================================
# 5. SINGLE QUANTUM EVALUATION
# ==========================================
print("Running the quantum circuit to extract expectation values...")
job = estimator.run([
    (ansatz, observables_x, optimal_theta),
    (ansatz, observables_y, optimal_theta),
    (ansatz, observables_z, optimal_theta)
])
result = job.result()

# Map the expectation values back to the 180 nodes
idx = 0
node_exp_map = {}
for r in result:
    for ev in r.data.evs:
        node_exp_map[idx] = ev
        idx += 1

# ==========================================
# 6. DECODING & LOCAL BIT-SWAP SEARCH
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        # Check if the edge crosses the partition boundary
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Step A: Thresholding expectation values into binary strings (>=0 is 1, <0 is 0)
cur_bits = []
for i in range(num_nodes):
    if node_exp_map[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

print("\nStarting local bit-swap search to finalize cut...")
best_cut = 0
best_bits = []

# Step B: 1-Bit Swap Search to polish the continuous relaxation
for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    
    # Swap the assignments of the nodes connecting this edge
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print(f"\n=====================================")
print(f"🎯 FINAL QUANTUM MAX-CUT SIZE: {best_cut}")
print(f"=====================================")

Loading dataset...
Successfully loaded 90 parameters from pce_optimal_theta_adam_reps4qbt12Continue (2).npy.
Running the quantum circuit to extract expectation values...

Starting local bit-swap search to finalize cut...

🎯 FINAL QUANTUM MAX-CUT SIZE: 5301.949528437715


In [4]:
def greedy_local_search(bits, graph, max_rounds=100):
    bits = list(bits)
    best_cut = calc_cut_size(graph, 
                              {i for i,b in enumerate(bits) if b==1},
                              {i for i,b in enumerate(bits) if b==0})
    improved = True
    rounds = 0
    while improved and rounds < max_rounds:
        improved = False
        rounds += 1
        for node in range(len(bits)):
            bits[node] ^= 1  # flip node
            p0 = {i for i,b in enumerate(bits) if b==1}
            p1 = {i for i,b in enumerate(bits) if b==0}
            new_cut = calc_cut_size(graph, p0, p1)
            if new_cut > best_cut:
                best_cut = new_cut
                improved = True
            else:
                bits[node] ^= 1  # flip back
    return bits, best_cut

# Apply AFTER your current decoder
best_bits_greedy, best_cut_greedy = greedy_local_search(best_bits, graph)
print(f"After greedy local search: {best_cut_greedy}")

After greedy local search: 6640.229185322797


In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False, return_exp=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    # Vectorized extraction of expectation values
    node_exp_map = np.concatenate([
        result[0].data.evs,
        result[1].data.evs,
        result[2].data.evs
    ])
    
    # Vectorized calculation of the Tanh values
    T = np.tanh(alpha_loss * node_exp_map)
    
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * T[edge0] * T[edge1]
        
    R = np.sum(T**2) / num_nodes
    regulation_term = beta_loss * v * (R**2)
    loss += regulation_term
    
    if not silent:
        # Reconstruct the dictionary for compatibility with the final decoding step
        exp_dict = {i: node_exp_map[i] for i in range(num_nodes)}
        experiment_result.append({"loss": loss, "exp_map": exp_dict})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    if return_exp:
        return loss, node_exp_map
    return loss

def silent_loss_func(x):
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. CHAIN RULE & PARAMETER SHIFT ENGINE
# ==========================================
def compute_dL_dE(E_vals):
    """Classical analytical derivative of the loss function w.r.t expectation values"""
    T = np.tanh(alpha_loss * E_vals)
    dT_dE = alpha_loss * (1.0 - T**2) # Derivative of tanh(alpha * E)

    dL_dT = np.zeros(num_nodes)
    
    # 1. Derivative of the Max-Cut edges
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        dL_dT[edge0] += w * T[edge1]
        dL_dT[edge1] += w * T[edge0]

    # 2. Derivative of the Regularization term
    R = np.sum(T**2) / num_nodes
    dL_dT += (4.0 * beta_loss * v / num_nodes) * R * T

    # Chain rule: dL/dE = dL/dT * dT/dE
    return dL_dT * dT_dE

def compute_gradient_param_shift(x, dL_dE):
    """Calculates exactly dE/dTheta using quantum shifts, and applies classical chain rule."""
    grad = np.zeros(len(x))
    shift = np.pi / 2.0
    pubs = []
    
    # Bundle all shifted parameters into one massive job for extreme efficiency
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += shift
        x_minus = x.copy()
        x_minus[i] -= shift
        
        pubs.extend([
            (ansatz, observables_x, x_plus),
            (ansatz, observables_y, x_plus),
            (ansatz, observables_z, x_plus),
            (ansatz, observables_x, x_minus),
            (ansatz, observables_y, x_minus),
            (ansatz, observables_z, x_minus)
        ])
        
    job = estimator.run(pubs)
    result = job.result()
    
    for i in range(len(x)):
        base_idx = i * 6
        # Combine the 3 sets of observables for the +pi/2 shift
        E_plus = np.concatenate([
            result[base_idx].data.evs,
            result[base_idx+1].data.evs,
            result[base_idx+2].data.evs
        ])
        # Combine the 3 sets of observables for the -pi/2 shift
        E_minus = np.concatenate([
            result[base_idx+3].data.evs,
            result[base_idx+4].data.evs,
            result[base_idx+5].data.evs
        ])
        
        # Parameter shift formula: 0.5 * (E_plus - E_minus)
        dE_dtheta = 0.5 * (E_plus - E_minus)
        
        # Final Chain Rule Application: dot product of dL/dE and dE/dtheta
        grad[i] = np.dot(dL_dE, dE_dtheta)
        
    return grad

# ==========================================
# 6. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_chkpt_200.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 7. EXACT ADAM OPTIMIZER (WITH COSINE DECAY & CHECKPOINTS)
# ==========================================
def adam_optimize(theta0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=500):
    print(f"Starting exact analytical optimization with Adam + Cosine Decay (Max LR={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    best_overall_loss = float('inf')
    best_overall_theta = np.copy(theta_i)
    
    current_loss = loss_func_estimator(theta_i, silent=False)
    if current_loss < best_overall_loss:
        best_overall_loss = current_loss
        best_overall_theta = np.copy(theta_i)
    
    for i in range(1, max_iter + 1):
        t += 1
        
        # Calculate Cosine Annealing Learning Rate
        lr_t = 0.005 + 0.5 * (lr - 0.005) * (1 + np.cos(np.pi * t / max_iter))
        
        print(f"\n--- Adam Step {i}/{max_iter} | LR: {lr_t:.6f} ---")
        
        # --- THE EXACT GRADIENT ENGINE ---
        # 1. Measure the current expectation values
        _, current_E_vals = loss_func_estimator(theta_i, silent=True, return_exp=True)
        # 2. Classically calculate the exact derivative of the loss function
        dL_dE = compute_dL_dE(current_E_vals)
        # 3. Apply the parameter shift rule on the quantum chip
        grad = compute_gradient_param_shift(theta_i, dL_dE)
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        adam_direction = m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Update with Cosine Annealed Step Size
        theta_next = theta_i - lr_t * adam_direction
        f_next = silent_loss_func(theta_next)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            current_loss = loss_func_estimator(theta_i, silent=False)
            if current_loss < best_overall_loss:
                best_overall_loss = current_loss
                best_overall_theta = np.copy(theta_i)
            break
            
        theta_i = theta_next
        current_loss = f_next 
        
        # Log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
        if current_loss < best_overall_loss:
            best_overall_loss = current_loss
            best_overall_theta = np.copy(theta_i)
            
        if i % 100 == 0:
            chkpt_filename = f"pce_optimal_theta_adam_chkpt_{i}.npy"
            np.save(chkpt_filename, best_overall_theta)
            print(f">>> CHECKPOINT SAVED: {chkpt_filename} | Best Loss: {best_overall_loss:.6f} <<<")
            
    print(f"\nAdam Optimization Complete! Absolute Best Loss Found: {best_overall_loss:.6f}")
    return best_overall_theta 

# Execute the exact Adam Optimizer
# Set max_lr to 0.1 or 0.2 depending on how aggressively you want to start
best_params = adam_optimize(
    theta0=initial_params,
    lr=0.1,        
    max_iter=700 
)

np.save("pce_optimal_theta_adam_FINAL.npy", best_params)

# ==========================================
# 8. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
print("Loading absolute best parameters for final bit-swap decoding...")
_ = loss_func_estimator(best_params, silent=False)

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_chkpt_200.npy...
Starting exact analytical optimization with Adam + Cosine Decay (Max LR=0.1)...
Iter 1: 15.504503

--- Adam Step 1/700 | LR: 0.100000 ---
Iter 2: -145.582054

--- Adam Step 2/700 | LR: 0.099998 ---
Iter 3: -215.687958

--- Adam Step 3/700 | LR: 0.099996 ---
Iter 4: -302.942345

--- Adam Step 4/700 | LR: 0.099992 ---
Iter 5: -364.279243

--- Adam Step 5/700 | LR: 0.099988 ---
Iter 6: -431.711946

--- Adam Step 6/700 | LR: 0.099983 ---
Iter 7: -507.548273

--- Adam Step 7/700 | LR: 0.099977 ---
Iter 8: -571.759520

--- Adam Step 8/700 | LR: 0.099969 ---
Iter 9: -602.999711

--- Adam Step 9/700 | LR: 0.099961 ---
Iter 10: -623.721129

--- Adam Step 10/700 | LR: 0.099952 ---
Iter 11: -649.583017

--- Adam Step 11/700 | LR: 0.099942 ---
Iter 12: -681.889599

--- Adam Step 12/700 | LR: 0.099931 ---
Iter 13: -727.247866

--- Adam Step 13/700 | LR: 0.099919 ---
Iter 14: -784.004661

--- Adam Step 14/7

KeyboardInterrupt: 